In [ ]:
from google.colab import files
files.upload()

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d salader/dogsvscats

In [ ]:
import zipfile
zip_ref = zipfile.ZipFile('/content/dogsvscats.zip', 'r')
zip_ref.extractall('/content')
zip_ref.close()

In [ ]:
import tensorflow as tf
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense, Conv2D, MaxPooling2D, Flatten, BatchNormalization, Dropout

In [ ]:
# GENERATORS #
train_ds=keras.utils.image_dataset_from_directory(
    directory = '/content/train',
    labels = 'inferred',
    label_mode = 'int',
    batch_size = 32,
    image_size = (256, 256)
)

validation_ds=keras.utils.image_dataset_from_directory(
    directory = '/content/test',
    labels = 'inferred',
    label_mode = 'int',
    batch_size = 32,
    image_size = (256, 256)
)

In [ ]:
#print(train_ds.class_names)

import os
print(os.listdir('/content/train'))
print(os.listdir('/content/test'))

In [ ]:
import numpy as np

all_labels = []

for images, labels in train_ds:
    all_labels.extend(labels.numpy())

print(np.unique(all_labels, return_counts=True))

In [ ]:
for images, labels in train_ds.take(1):
    print("Image shape:", images.shape)
    print("Labels:", labels.numpy())

In [ ]:
# NORMALIZE #
def process(image, label):
    image = tf.cast(image/255. ,tf.float32)
    return image, label

train_ds = train_ds.map(process)
validation_ds = validation_ds.map(process)


In [ ]:
# Create CNN model
model = Sequential()

model.add(Conv2D(32, kernel_size=(3, 3), padding='valid', activation='relu', input_shape=(256, 256, 3)))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2, 2), strides=2, padding='valid'))

model.add(Conv2D(64, kernel_size=(3, 3), padding='valid', activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2, 2), strides=2, padding='valid'))

model.add(Conv2D(128, kernel_size=(3, 3), padding='valid', activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2, 2), strides=2, padding='valid'))

model.add(Flatten())

model.add(Dense(128, activation='relu'))
model.add(Dropout(0.1))
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.1))
model.add(Dense(1, activation='sigmoid'))


# model = tf.keras.Sequential([

#     tf.keras.layers.Conv2D(32, (3,3), activation='relu', input_shape=(256,256,3)),
#     tf.keras.layers.MaxPooling2D(),

#     tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
#     tf.keras.layers.MaxPooling2D(),

#     tf.keras.layers.Conv2D(128, (3,3), activation='relu'),
#     tf.keras.layers.MaxPooling2D(),

#     tf.keras.layers.Flatten(),

#     tf.keras.layers.Dense(128, activation='relu'),
#     tf.keras.layers.Dropout(0.5),

#     tf.keras.layers.Dense(1, activation='sigmoid')
# ])

model.summary()

In [ ]:
# COMPILATION #
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# TRAINING #
history=model.fit(train_ds,epochs=15,validation_data=validation_ds)

In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history['accuracy'], color='red', label='train')
plt.plot(history.history['val_accuracy'], color='blue', label='validation')
plt.legend()
plt.show()

In [ ]:
import cv2

test_img=cv2.imread("/content/download (1).jpg")

plt.imshow(test_img)
#plt.show()

In [ ]:
test_img = cv2.imread("/content/download.jpg")
test_img = cv2.cvtColor(test_img, cv2.COLOR_BGR2RGB)

test_img = cv2.resize(test_img, (256, 256))
test_img = test_img / 255.0

test_input = np.expand_dims(test_img, axis=0)

prediction = model.predict(test_input)

if prediction[0][0] > 0.5:
    print("Dog 🐶")
else:
    print("Cat 🐱")

In [ ]:
model.save("cat_dog_cnn.keras")
from google.colab import files
files.download("cat_dog_cnn.keras")